# OMIP runs: SST percentile method & perturbation percentiles

**What's in this notebook**

- Applies the percentile / cold-warm method to OMIP model SST.
- Loads multiple OMIP models; computes percentile trends and confidence intervals.
- Perturbation-SST percentiles.

_Split from `omip_compare.ipynb` (percentiles). Shared helpers in `functions.py`; outputs stripped — re-run to regenerate figures._


In [ ]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib

%matplotlib inline


In [ ]:
import sys, os
# make this repo's functions.py importable from any subfolder
_p = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_p, 'functions.py')):
        sys.path.insert(0, _p); break
    _p = os.path.dirname(_p)
from functions import *  # shared helper functions


In [ ]:
# Load variables produced by earlier (compute/loader) notebooks.
# Run those first; harmless if there's nothing saved yet.
load_state(globals())


# Apply percentile method

- scatter plot with east-west x axis and cold-warm y axis - trend
- trend of each percentile plot like in paper

### Load other OMIP data

In [ ]:
models = ['CanESM5', 'EC', 'FGOALS','FIO', 'MRI', 'NorESM2']
#, 'CMCC'
# pull .nc files which contain the model name 

directory_path = '/Users/isabelaconde/Desktop/data/omip/gr/'
# loop through the models and open the files
files = [f for f in os.listdir(directory_path) if f.endswith('.nc')]


models = [f.split('_')[1] for f in files]

model_datasets = {}
for i,file in enumerate(files):

    model_datasets[models[i]] = xr.open_dataset(directory_path+file,decode_times=False)
    print('' + models[i] + ' dataset loaded')
    
model_datasets# model_datasets

In [ ]:
# mask pacific

def pacific_mask(sst):
    LonsPts_pos = [130,120,110,110,110,260,270,270,277,291,291,130] #-np.ones(13)*180
    LatsPts_pos =[-20,-10,-5,10,20,20,15,15,10,5,-20,-20]
    
    x_pixel_pos = LonsPts_pos
    y_pixel_pos = LatsPts_pos
    
    temp_list = []
    for a, b in zip(x_pixel_pos, y_pixel_pos):
        temp_list.append([a, b])
    
    polygon = np.array(temp_list)
    
    xv, yv = sst.lon.values, sst.lat.values
    lon2d, lat2d = np.meshgrid(xv, yv)
    points = np.hstack((lon2d.reshape((-1,1)), lat2d.reshape((-1,1))))
    path = matplotlib.path.Path(polygon)
    mask = path.contains_points(points)
    mask.shape = lon2d.shape
    mask_bit_pos = mask*np.ones_like(lon2d)
    
    mask_mods = mask_bit_pos#+mask_bit_neg
    
    mask_mods[mask_mods==0] = np.nan
    return mask_mods


# mask models
def mask_models(model_datasets):
    for model, dataset in model_datasets.items():
        print('Masking ' + model)
        dataset['sst'] = dataset['sst'].where(pacific_mask(dataset['sst']) == 1)
        dataset['sst'].attrs['long_name'] = 'Sea Surface Temperature'
        dataset['sst'].attrs['units'] = 'C'
        print(model + ' masked')
    return model_datasets

# mask models
model_datasets = mask_models(model_datasets)

# Percentiles 

In [ ]:
# load percentile data processed in gadi 

mod_labs = ['MRI-ESM2-0', 'FGOALS-f3-L','EC-EARTH3','ACCESS-OM2-JRA55-cycle6_iaf','ACCESS-OM2-ERA5']

save_directory = '/Users/isabelaconde/Desktop/data/omip/sst_percentiles/'

file_paths = [f"{save_directory}SST_percentile_{model}_1982-2018_gr.nc" for model in mod_labs]

# some datasets have cftime - change in loop below
import datetime
td = datetime.datetime.strptime("1982-01-01", "%Y-%m-%d")
k = 444
d = pd.date_range(td, periods=k, freq='MS')

dict_percentiles_omip = {}

for i, file in enumerate(file_paths):
    dict_percentiles_omip[mod_labs[i]] = xr.open_dataarray(file).assign_coords({'time': d})



In [ ]:

# apply function to each model

percentiles_omip_trend = {}
for i, da in enumerate(mod_labs):
    
    percentiles_omip_trend[da]=find_trend_percentile(dict_percentiles_omip[da].sel(time = slice('1982','2017')))
    print('trend calculated for ' + mod_labs[i])

for i in range(len(mod_labs)):
    # plot trends of highest and lowest model 
    percentiles_omip_trend[mod_labs[i]].plot(label = mod_labs[i])  



plt.legend()

In [ ]:
# load percentile trend 1982--2018
# make DA_all 
# load percentile trend 1982--2018

trend_percentile = xr.open_datatree('/Users/isabelaconde/Desktop/data/dt_cmip_percentiles_trend_1982-2018.nc')

# find mods
# Get ensemble names - yes so you can call on it later
models =['CESM1-CAM5',
 'CanESM2',
 'GFDL-ESM2M',
 'MPI-ESM',
 'ACCESS-ESM1-5',
 'CanESM5',
 'EC-Earth3',
 'IPSL-CM6A-LR',
 'MIROC-ES2L',
 'MIROC6',
 'MPI-ESM1-2-LR',
 'CESM2',
 'GFDL-SPEAR-MED']

data_tree=trend_percentile

mods = {}
for model_name, model_data in data_tree.items():
    en=[]
    for ensemble_name, ensemble_data in data_tree[model_name].items():
        en.extend([str(ensemble_name)])
    
    mods[str(model_name)]=en

    
trend_dt=trend_percentile
model_array=[]
da=[]

for m in range(len(models)):
    sources = mods[models[m]][0:len(mods[models[m]])]
    # da=[] # initialise array for grad values
    for i in range(len(sources)):
        da.extend(trend_dt[models[m]].grad.values)

DA_all = xr.DataArray(np.squeeze(da), dims=['run','area'], coords={'run': np.linspace(0, 1, 519), 'area': np.linspace(0, 1, 100)})


In [ ]:
# DA_all= xr.open_dataarray('/Users/isabelaconde/Desktop/data/final_data/da_all_cmip56_runs.nc')

def calculate_confidence_intervals_all(data, n):
    mean = np.empty(n)
    lower = np.empty(n)
    upper = np.empty(n)

    for i in range(data.shape[0]):
        mean[i] = np.mean(data[i])
        lower[i] = scoreatpercentile(data[i],10)
        upper[i] = scoreatpercentile(data[i],90)

    return mean, lower, upper

# Plot for first three models
num=100
data = DA_all.values.T * 10 * 12
mean, lower, upper = calculate_confidence_intervals_all(data, num)
df = pd.DataFrame({'mean': DA_all.mean(dim='run').values * 10 * 12, 
                   'min': DA_all.min(dim='run').values * 10 * 12, 
                   'max': DA_all.max(dim='run').values * 10 * 12, 
                   'lower': lower, 'upper': upper,
                   'area': np.linspace(0, 1, 100)})

noaa_percentile_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/cmip_compare/noaa_percentile_trend.nc')
cobe_percentile_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/cmip_compare/cobe_percentile_trend.nc')
had_percentile_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/cmip_compare/ersst_percentile_trend.nc')

obs_mean = (noaa_percentile_trend+cobe_percentile_trend+had_percentile_trend)/3


In [ ]:
from xhistogram.xarray import histogram

def percentiles_xhist(data): # find percentiles of SST using xhistogram
    hist_T = data.squeeze()
    weights = np.cos(np.deg2rad(hist_T.lat))
    times = hist_T['time']
    const_area=np.r_[0:1:0.01]
    temps = np.r_[12:34:0.5] # define temperatures for binning 
    
    temps_interp = np.empty([len(times),len(const_area)])
    
    for i in np.r_[0:len(times):1]:  
        h=histogram(hist_T[i],bins=temps,weights=weights).cumsum()
        hh = (h - h[0])/(h[-1]-h[0])
        temps_interp[i]=np.interp(const_area,hh,temps[1:])
    
    
    sst_area= xr.DataArray(temps_interp,coords={'time':times,'area':const_area})

    return(sst_area)

def pacific_mask(sst):

    LonsPts_pos = [130,120,110,110,110,260,270,270,277,291,291,130] #-np.ones(13)*180
    LatsPts_pos =[-20,-10,-5,10,20,20,15,15,10,5,-20,-20]
    
    x_pixel_pos = LonsPts_pos
    y_pixel_pos = LatsPts_pos
    
    temp_list = []
    for a, b in zip(x_pixel_pos, y_pixel_pos):
        temp_list.append([a, b])
    
    polygon = np.array(temp_list)
    
    xv, yv = sst.lon.values, sst.lat.values
    lon2d, lat2d = np.meshgrid(xv, yv)
    points = np.hstack((lon2d.reshape((-1,1)), lat2d.reshape((-1,1))))
    path = matplotlib.path.Path(polygon)
    mask = path.contains_points(points)
    mask.shape = lon2d.shape
    mask_bit_pos = mask*np.ones_like(lon2d)

    mask_mods = mask_bit_pos
    
    mask_mods[mask_mods==0] = np.nan
    return mask_mods


def trend_obs_percentile(data): # find trend of percentile data
    time_axis = np.arange(data.sizes['time'])
    trends = np.empty(data.sizes['area'])
    for i in range(data.sizes['area']):
        y = data[:, i].values
        trends[i] = np.polyfit(time_axis, y, 1)[0]  # slope

    return xr.DataArray(trends, coords={'area': data.area}, dims='area', name='sst_trend')


noaa_sst = xr.open_dataset('/Users/isabelaconde/Desktop/data/conde2025/NOAA_sst.mon.mean.nc').sst.sel(time = slice('1982','2024')).sel(lat = slice(-80,80))
noaa_sst_pacific = (pacific_mask(noaa_sst)*noaa_sst).sel(lat=slice(-20,20), lon = slice(100,300))
noaa_sst_percentiles = percentiles_xhist(noaa_sst_pacific)
noaa_sst_trend = trend_obs_percentile(noaa_sst_percentiles)


In [ ]:
control_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_control.nc')
era5_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_era5.nc')
perturb_percent_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_merra2_percentile_trend.nc')
# perturb_percent_era5_uas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_percentile_era5_uas_trend.nc')
perturb_percent_era5_uas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/1deg_jra55_iaf_era5_uas_trend_percentile.nc')
perturb_percent_era5_uas_vas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_uas_vas.nc')
perturb_percent_era5_huss_tas_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_huss_tas.nc')
perturb_percent_era5_rsds_rlds_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_rsds_rlds.nc')
perturb_percent_era5_winds = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_era5_winds.nc')
perturb_percent_all_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/ACCESS-OM2_runs/perturb_percentile_trend_all_trend.nc')
perturb_percent_winds_mean_trend = xr.open_dataarray('/Users/isabelaconde/Desktop/data/omip/processed/perturb_percentile_trend_uas_vas_mean_trend.nc')


In [ ]:
# plot to see what it looks like 
fig, axes = plt.subplots(figsize=(10, 6))
# array of 5 different colourblind okay colours
autumn_colors = [plt.cm.autumn(i) for i in np.linspace(0, 1, 4)]
colors = autumn_colors + ['blue']  # add blue at the end


for i, da in enumerate(mod_labs):
    (percentiles_omip_trend[da]*12*10).plot(ax=axes, label=mod_labs[i], color=colors[i], alpha =0.5)

(control_percent_trend*12*10).plot(ax=axes, label='ACCESS-OM2-JRA55-iaf control', color='lime')
(perturb_percent_trend*12*10).plot(ax=axes, label='ACCESS-OM2-JRA55-iaf MERRA2 uas', color='firebrick')
(perturb_percent_era5_uas_trend*12*10).plot(ax=axes, label='ACCESS-OM2-JRA55-iaf ERA5 uas', color='cyan')
(perturb_percent_era5_uas_vas_trend*12*10).plot(ax=axes, label='ACCESS-OM2-JRA55-iaf ERA5 uas & vas', color='blue')





sns.lineplot(x="area", y="mean", data=df, ax=axes, color='grey', label='CMIP 5/6 Multi-Model Mean')
axes.fill_between(df['area'], df['upper'], df['lower'], alpha=0.2, color='red', label='0.1\u20130.9 Decile Range')
# axes.fill_between(df['area'], df['max'], df['min'], alpha=0.2, color='grey', label='Min\u2013Max Range')
axes.plot(noaa_sst_trend.area.values, noaa_sst_trend.values * 10 * 12, color='k', label='NOAA OISST')
# # 


# set legend
# Modify x-axis labels to represent percentages
axes.set_xticks(np.linspace(0, 1, 11))  # 11 ticks for 0%, 10%, 20%, ..., 100%
axes.set_xticklabels(['{}%'.format(int(x * 100)) for x in np.linspace(0, 1, 11)])

# Add grid
axes.grid(True)


# Add legend
axes.legend(loc = 'lower right',fontsize=9)
axes.set_xlim(0, 1)
axes.set_ylabel(r'$^{\circ}$C (10 yr)$^{-1}$')
axes.set_xlabel('Area')
plt.title('Trend at Each SST Precentile (1983\u20132017)',fontsize=14)
plt.savefig(os.path.expanduser("~/Desktop/percents_trend_omip_comparison.pdf"), bbox_inches='tight')



In [ ]:
fig, axes = plt.subplots(figsize=(10, 6))

# -------------------------
# OMIP SHADED REGION (exclude last model)
# -------------------------
omip_models = mod_labs[:-1]          # all except last
special_mod = mod_labs[-1]           # last model (plot separately)

# Stack all OMIP models
omip_stack = xr.concat(
    [percentiles_omip_trend[m] for m in omip_models],
    dim='model'
) * 12 * 10

# Ensemble stats
omip_min  = omip_stack.min(dim='model')
omip_max  = omip_stack.max(dim='model')
omip_mean = omip_stack.mean(dim='model')

# Shaded region
axes.fill_between(
    omip_stack.area,
    omip_min,
    omip_max,
    color='cadetblue',
    alpha=0.3,
    label='OMIP Model Spread'
)

(era5_percent_trend * 12 * 10).plot(
    ax=axes,
    color='darkviolet',
    label='ERA5',
    lw=1.5
)

# -------------------------
# ACCESS-OM2 runs
# -------------------------
(control_percent_trend*12*10).plot(ax=axes, label='JRA55 control', color='firebrick',lw=1)
# (perturb_percent_era5_uas_trend*12*10).plot(ax=axes, label='JRA55 with ERA5 uas trend', color='cyan')
(perturb_percent_era5_rsds_rlds_trend*12*10).plot(ax=axes, label='JRA55 with ERA5 rsds & rlds trend', color='orangered',lw=1)
((perturb_percent_era5_huss_tas_trend)*12*10).plot(ax=axes, label='JRA55 with ERA5 huss & tas trend', color='limegreen')
((perturb_percent_all_trend)*12*10).plot(ax=axes, label='JRA55 with ERA5 all-ish trends', color='orange',lw = 1.5)

(perturb_percent_era5_uas_vas_trend*12*10).plot(ax=axes, label='JRA55 with ERA5 uas & vas trend', color='cyan')
((perturb_percent_winds_mean_trend)*12*10).plot(ax=axes, label='JRA55 with ERA5 uas & vas mean and trend', color='b')
((perturb_percent_era5_winds)*12*10).plot(ax=axes, label='JRA55 with ERA5 winds trend', color='violet')



# -------------------------
# CMIP + envelope
# -------------------------
sns.lineplot(x="area", y="mean", data=df, ax=axes, color='grey', label='CMIP 5/6 Multi-Model Mean')
axes.fill_between(df['area'], df['upper'], df['lower'], alpha=0.2, color='red',
                  label='0.1–0.9 Decile Range')

# NOAA observational line
axes.plot(noaa_sst_trend.area.values, noaa_sst_trend.values * 10 * 12,
          color='k', label='NOAA OISST')

# -------------------------
# Formatting
# -------------------------
axes.set_xticks(np.linspace(0, 1, 11))
axes.set_xticklabels([f'{int(x*100)}%' for x in np.linspace(0, 1, 11)])
axes.grid(True)
axes.legend(loc='lower right', fontsize=9)
axes.set_xlim(0, 1)
axes.set_ylabel(r'$^{\circ}$C (10 yr)$^{-1}$')
axes.set_xlabel('Area')
plt.title('Trend at Each SST Percentile (1982–2017)', fontsize=14)
outpath = "/Users/isabelaconde/Desktop/trend_percentile.pdf"
fig.savefig(outpath, format="pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(figsize=(10, 6))

# -------------------------
# OMIP SHADED REGION (exclude last model)
# -------------------------
omip_models = mod_labs[:-1]          # all except last
special_mod = mod_labs[-1]           # last model (plot separately)

# Stack all OMIP models
omip_stack = xr.concat(
    [percentiles_omip_trend[m] for m in omip_models],
    dim='model'
) * 12 * 10

# Ensemble stats
omip_min  = omip_stack.min(dim='model')
omip_max  = omip_stack.max(dim='model')
omip_mean = omip_stack.mean(dim='model')

# Shaded region
axes.fill_between(
    omip_stack.area,
    omip_min,
    omip_max,
    color='cadetblue',
    alpha=0.3,
    label='OMIP Model Spread'
)

(era5_percent_trend * 12 * 10).plot(
    ax=axes,
    color='darkviolet',
    label='ERA5',
    lw=1.5
)

# -------------------------
# ACCESS-OM2 runs
# -------------------------
(control_percent_trend*12*10).plot(ax=axes, label='JRA55 control', color='firebrick')
# (perturb_percent_era5_uas_trend*12*10).plot(ax=axes, label='JRA55 with ERA5 uas trend', color='cyan')
(perturb_percent_era5_rsds_rlds_trend*12*10).plot(ax=axes, label='JRA55 with ERA5 rsds & rlds trend', color='orangered')
((perturb_percent_era5_huss_tas_trend)*12*10).plot(ax=axes, label='JRA55 with ERA5 huss & tas trend', color='limegreen')
# ((perturb_percent_all_trend)*12*10).plot(ax=axes, label='JRA55 with ERA5 all-ish trends', color='orange',lw = 1.5)

(perturb_percent_era5_uas_vas_trend*12*10).plot(ax=axes, label='JRA55 with ERA5 uas & vas trend', color='cyan')
((perturb_percent_winds_mean_trend)*12*10).plot(ax=axes, label='JRA55 with ERA5 uas & vas mean and trend', color='b')
((perturb_percent_era5_winds)*12*10).plot(ax=axes, label='JRA55 with ERA5 winds trend', color='violet')



# -------------------------
# CMIP + envelope
# -------------------------
sns.lineplot(x="area", y="mean", data=df, ax=axes, color='grey', label='CMIP 5/6 Multi-Model Mean')
axes.fill_between(df['area'], df['upper'], df['lower'], alpha=0.2, color='red',
                  label='0.1–0.9 Decile Range')

# NOAA observational line
axes.plot(noaa_sst_trend.area.values, noaa_sst_trend.values * 10 * 12,
          color='k', label='NOAA OISST')

# -------------------------
# Formatting
# -------------------------
axes.set_xticks(np.linspace(0, 1, 11))
axes.set_xticklabels([f'{int(x*100)}%' for x in np.linspace(0, 1, 11)])
axes.grid(True)
axes.legend(loc='lower right', fontsize=9)
axes.set_xlim(0, 1)
axes.set_ylabel(r'$^{\circ}$C (10 yr)$^{-1}$')
axes.set_xlabel('Area')
plt.title('Trend at Each SST Percentile', fontsize=14)
outpath = "/Users/isabelaconde/Desktop/trend_percentile.pdf"
fig.savefig(outpath, format="pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(figsize=(10, 6))

# -------------------------
# CMIP + envelope
# -------------------------
sns.lineplot(x="area", y="mean", data=df, ax=axes, color='grey', label='CMIP 5/6 Multi-Model Mean')
axes.fill_between(df['area'], df['upper'], df['lower'], alpha=0.2, color='red',
                  label='0.1–0.9 Decile Range')

# NOAA observational line
axes.plot(noaa_sst_trend.area.values, noaa_sst_trend.values * 10 * 12,
          color='k', label='NOAA OISST')

# -------------------------
# OMIP SHADED REGION (exclude last model)
# -------------------------
omip_models = mod_labs[:-1]          # all except last
special_mod = mod_labs[-1]           # last model (plot separately)

# Stack all OMIP models
omip_stack = xr.concat(
    [percentiles_omip_trend[m] for m in omip_models],
    dim='model'
) * 12 * 10

# Ensemble stats
omip_min  = omip_stack.min(dim='model')
omip_max  = omip_stack.max(dim='model')
omip_mean = omip_stack.mean(dim='model')

# Shaded region
axes.fill_between(
    omip_stack.area,
    omip_min,
    omip_max,
    color='cadetblue',
    alpha=0.3,
    label='OMIP Model Spread'
)

(era5_percent_trend * 12 * 10).plot(
    ax=axes,
    color='darkviolet',
    label='ERA5 ACCESS-OM2',
    lw=1.5
)

# -------------------------
# ACCESS-OM2 runs
# -------------------------
(control_percent_trend * 12 * 10).plot(ax=axes, label='JRA55-do ACCESS-OM2', color='firebrick')
# (perturb_percent_era5_uas_trend*12*10).plot(ax=axes, label='JRA55 with ERA5 uas trend', color='cyan')
# (perturb_percent_era5_uas_vas_trend * 12 * 10).plot(ax=axes, label='JRA55-do ACCESS-OM2 with ERA5 uas & vas trend', color='cyan')
# (perturb_percent_era5_winds * 12 * 10).plot(ax=axes, label='JRA55-do ACCESS-OM2 with exact ERA5 winds', color='violet')
# (perturb_percent_era5_rsds_rlds_trend * 12 * 10).plot(ax=axes, label='JRA55-do ACCESS-OM2 with ERA5 rsds & rlds trend', color='orange')
# (perturb_percent_era5_huss_tas_trend * 12 * 10).plot(ax=axes, label='JRA55-do ACCESS-OM2 with ERA5 huss & tas trend', color='limegreen')
# (perturb_percent_all_trend * 12 * 10).plot(ax=axes, label='JRA55-do ACCESS-OM2 with ERA5 all-ish trends', color='red',lw = 2)

# -------------------------

# -------------------------
# Formatting
# -------------------------
axes.set_xticks(np.linspace(0, 1, 11))
axes.set_xticklabels([f'{int(x*100)}%' for x in np.linspace(0, 1, 11)])
axes.grid(True)
axes.set_xlim(0, 1)
axes.set_ylabel(r'$^{\circ}$C (10 yr)$^{-1}$')
axes.set_xlabel('Area')
axes.set_title('Trend at Each SST Percentile (1982–2017)', fontsize=14)

# -------------------------
# Legend outside (right)
# -------------------------
fig.subplots_adjust(right=0.78)  # leave room for legend on the right
axes.legend(loc='lower right', fontsize=9)

# axes.legend(
#     loc='center left',
#     bbox_to_anchor=(1.02, 0.8),
#     fontsize=9,
#     frameon=False
# )

# plt.show()

outpath = "/Users/isabelaconde/Desktop/Trend_at_each_SST_percentile_1982-2017.pdf"
fig.savefig(outpath, format="pdf", bbox_inches="tight")

#### perturbation sst percentiles

In [ ]:
sst_perturb = xr.open_dataarray('/Users/isabelaconde/Desktop/data/SST_perturb.nc').rename({'xt_ocean':'lon', 'yt_ocean':'lat'})

#lons from 0 to 360
sst_perturb['lon'] = (sst_perturb['lon']+360) % 360
#sort by lon 
sst_perturb = sst_perturb.sortby(sst_perturb['lon'])
# sst_perturb 

# .sel(lat=slice(-20, 20), lon=slice(110, 291))
sst_perturb = sst_perturb.sel(lat=slice(-20, 20), lon=slice(110, 291))


In [ ]:
# sst_perturb.isel(time=0).plot()


In [ ]:
# load packages 
import numpy as np
import xarray as xr
from xarray import DataTree
import matplotlib
import math

# from xmip.postprocessing import _parse_metric

import cartopy.crs as ccrs

# import xskillscore as xs
import matplotlib.pyplot as plt
%matplotlib inline
import calendar
from matplotlib.lines import Line2D

import os
import re
import seaborn as sns
import pandas as pd

import matplotlib.gridspec as gridspec

from scipy.stats import norm

import matplotlib.patches as patches
from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec


from scipy.stats import scoreatpercentile

import xesmf as xe


In [ ]:
# mask pacific with mask_pacific funciton
sst_perturb_pa = sst_perturb*pacific_mask(sst_perturb)
# find percentiles
perturb_percentiles = quantiles_xhist(sst_perturb).sel(time = slice('1982','2018'))
perturb_percentiles

In [ ]:
perturb_percentile_trend = find_trend_percentile(perturb_percentiles)
perturb_percentile_trend

In [ ]:
# load gridded data processed in gadi 
from datetime import datetime


mod_labs = ['MRI-ESM2-0', 'FGOALS-f3-L','EC-EARTH3','ACCESS-OM2-JRA55-cycle6_iaf','ACCESS-OM2-ERA5']

save_directory = '/Users/isabelaconde/Desktop/data/omip/tos/'

file_paths = [f"{save_directory}tos_global_{model}_1982-2018_gr.nc" for model in mod_labs]
gr_omip = {}

td = datetime.strptime("1982-01-01", "%Y-%m-%d")
k = 444
d = pd.date_range(td, periods=k, freq='MS')


for i, path in enumerate(file_paths):
    ds = xr.open_dataarray(path)
    ds = ds.assign_coords({'time': d})
    gr_omip[mod_labs[i]] = xr.open_dataarray(path)

gr_omip['MRI-ESM2-0']

In [ ]:
def find_trend_values(dt):

    #Input: xarray array with time, lat, lon coordinates and a variable to calculate the trend
    #Output: xarray array with the trend values for each grid point
    
    time, lat, lon = dt['time'].values, dt['lat'].values, dt['lon'].values
    sst_reshaped = dt.values.reshape(len(time), -1)
    degree = 1

    gradient_values = np.empty_like(sst_reshaped[0])
    for i in range(len(sst_reshaped[0])):
        y = sst_reshaped[:, i]
        coefficients = np.polyfit(np.arange(len(time)), y, degree)
        gradient_values[i] = coefficients[-2]  # Coefficient for x term (slope)

    gradient_values = gradient_values.reshape(lat.shape[0], lon.shape[0])
    gradient_da = xr.DataArray(gradient_values, coords={'lat': lat, 'lon': lon}, dims=['lat', 'lon'])
    return gradient_da 

# find trend of gridded data    
gr_omip_trend = []
for i, da in enumerate(gr_omip):
    # make sure time is datetime
    print(da)
    
    gr_omip_trend.append(find_trend_values(gr_omip[da].sel(lon = slice(110, 290)).sel(lat = slice(-20, 20))))
    print('trend calculated for ' + mod_labs[i])


In [ ]:
noaa_trend= xr.open_dataarray('/Users/isabelaconde/Desktop/data/ma_ncfiles/noaa_trend.nc')
noaa_trend = noaa_trend.gradient.drop('rmse').drop('gradient').drop('zlev').sel(lon = slice(110, 290)).sel(lat = slice(-20, 20))
noaa_trend
gr_omip_trend.append(noaa_trend)
# gr_omip_trend[

In [ ]:

# -------------------------
# Data + labels (2 x 3)
# -------------------------
das = gr_omip_trend
labels = [
    "MRI-ESM2-0",
    "FGOALS-f3-L",
    "EC-EARTH3",
    "ACCESS-OM2-JRA55-cycle6",
    "ACCESS-OM2-ERA5",
    "NOAA-OISST",
]

# --- common colour limits (fixed, symmetric) ---
vmin, vmax = -0.35, 0.35

# --- map setup ---
proj = ccrs.PlateCarree(central_longitude=180)
pc0  = ccrs.PlateCarree()
extent = [110, 300, -20, 20]

# -------------------------
# Figure layout (2 rows x 3 cols)
# -------------------------
fig = plt.figure(figsize=(17, 5))
gs = GridSpec(2, 3, figure=fig, wspace=0.03, hspace=-0.07)

axes = [fig.add_subplot(gs[r, c], projection=proj) for r in range(2) for c in range(3)]

ims = []
for ax, title, da in zip(axes, labels, das):
    im = ax.pcolormesh(
        da.lon, da.lat, (da * 10 * 12),
        cmap="RdBu_r", vmin=vmin, vmax=vmax, transform=pc0
    )
    ax.set_extent(extent, crs=pc0)
    ax.coastlines()
    ax.set_title(title, fontsize=12)
    ims.append(im)

# gridlines + labels (only left column has y labels; only bottom row has x labels)
gls = []
for ax in axes:
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False
    gls.append(gl)

for i, gl in enumerate(gls):
    r, c = divmod(i, 3)
    if c != 0:
        gl.left_labels = False
    if r != 1:
        gl.bottom_labels = False

# single shared colourbar
cax = fig.add_axes([0.92, 0.20, 0.012, 0.60])
fig.colorbar(ims[0], cax=cax, label=r"$^{\circ}$C (10 yr)$^{-1}$")

plt.tight_layout(rect=[0, 0, 0.90, 1], pad=0.2)
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(17, 8))
axes = axes.flatten()  # Make it 1D so you can index with i
labels = ['MRI-ESM2-0', 'FGOALS-f3-L','EC-EARTH3','ACCESS-OM2-JRA55-cycle6','ACCESS-OM2-ERA5','NOAA-GFDL']
for i, da in enumerate(gr_omip_trend):
    im = (da*10*12).plot(ax=axes[i], add_colorbar=False, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    axes[i].set_title(labels[i])
        # Add colorbar only to axes[1] and axes[4] (i.e. subplots 2 and 5)
    if i in [2, 5]:
        cbar = fig.colorbar(im, ax=axes[i], orientation='vertical')


In [ ]:
gr_omip_trend[-3].plot()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import numpy as np
import matplotlib.gridspec as gridspec
fig = plt.figure(figsize=(22, 18))
gs = GridSpec(2, 3, figure=fig)  # 2x3 grid
gs.update(wspace=0.1, hspace=-0.8)  # reduce the spacing between rows with hspace=0.05

labels = ['MRI-ESM2-0', 'FGOALS-f3-L', 'EC-EARTH3',
          'ACCESS-OM2-JRA55-cycle6', 'ACCESS-OM2-ERA5', 'NOAA OISST']

for i, da in enumerate(gr_omip_trend):
    ax = fig.add_subplot(gs[i // 3, i % 3])

    # Setup Basemap
    m = Basemap(ax=ax, projection='cyl',
                llcrnrlat=-20, urcrnrlat=20, llcrnrlon=110, urcrnrlon=290, area_thresh=10000)
    m.drawcoastlines()
    m.drawparallels(np.arange(-20, 21, 10), labels=[1, 0, 0, 0], fontsize=8)
    m.drawmeridians(np.arange(120, 291, 20), labels=[0, 0, 0, 1], fontsize=8)



    # Plot with no colorbar
    m.imshow(da * 10 * 12, cmap='RdBu_r', vmin=-0.3, vmax=0.3)

    cbar_ax = fig.add_axes([0.92, 0.39, 0.01, 0.2])  # Adjust the position and size to make it smaller
    sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.3, vmax=0.3))
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax, label=r'$^{\circ}$C (10 yr)$^{-1}$')


    ax.set_title(labels[i], fontsize=12)

plt.savefig(os.path.expanduser("~/Desktop/grid_trend_omip_comparison.pdf"), bbox_inches='tight')


In [ ]:
# Save this notebook's variables so later (plotting) notebooks can load them.
save_state(globals())
